<a href="https://colab.research.google.com/github/userNOTfound-bot/Ollama-Google-Colab/blob/main/Ollama_Google_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚠️ You must have ***Ollama*** installed in your system.
If you didn't had that, then follow these steps to install that.

Installation step:
1. go to Ollama official website or use this redirect link https://ollama.com
2. Click on the download button and choose your OS, then download & install.

***Installing Ollama in your system is done.***

Now follow these steps in below 👇

In [ ]:
# @title Install Ollama + Cloudflared, then start tunnel

# Step 1: Install Ollama
import subprocess
subprocess.run('curl -fsSL https://ollama.ai/install.sh | sh', shell=True, check=True)

# Step 2: Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb
!cloudflared -v

# Step 3: Configure environment
import os
os.environ['OLLAMA_HOST'] = '0.0.0.0'
os.environ.update({'LD_LIBRARY_PATH': '/usr/lib64-nvidia'})

import asyncio

# Step 4: Run Ollama + Cloudflared together
async def run(cmd):
    '''
    run is a helper function to run subcommands asynchronously.
    '''
    print('>>> starting', *cmd)
    p = await asyncio.subprocess.create_subprocess_exec(
        *cmd,
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE,
    )

    async def pipe(lines):
        async for line in lines:
            print(line.strip().decode('utf-8'))

    await asyncio.gather(
        pipe(p.stdout),
        pipe(p.stderr),
    )

# Print lines utf-8 encoded
await asyncio.gather(
    run(['/usr/local/bin/ollama', 'serve']),
    run(['cloudflared', 'tunnel', '--url', 'http://localhost:11434']),
)
